<table align="left">
  <td>
    <a href="https://colab.research.google.com/" target="_parent">
      <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Abrir en Colab" title="Abrir y ejecutar en Google Colaboratory"/>
    </a>
  </td>
  <td>
    <a target="_blank" href="https://kaggle.com/kernels/welcome">
      <img src="https://kaggle.com/static/images/open-in-kaggle.svg" alt="Abrir en Kaggle" title="Abrir y ejecutar en Kaggle"/>
    </a>
  </td>
</table>

<br><br>

# Validación, selección de modelos, KNN, SVM e imbalance

**Curso:** Aprendizaje de Máquina Aplicado — Electiva  
**Session 4:** Validación, selección de modelos, KNN, SVM e imbalance  
**Universidad:** EAFIT  
**Profesor:** Marco Teran  

---

## Propósito del notebook

Este notebook materializa el contrato metodológico de la sesión:

1. diseñar un protocolo de validación honesto;
2. comparar modelos clásicos no lineales bajo el mismo flujo;
3. entender KNN desde distancia, scaling y elección de \(k\);
4. entender SVM desde margen, \(C\), kernel RBF y \(\gamma\);
5. hacer HPO sin tocar test;
6. manejar imbalance como decisión metodológica, no como receta;
7. seleccionar threshold con evidencia out-of-fold;
8. evaluar test solo al final;
9. introducir Gaussian Processes como extensión breve de incertidumbre;
10. transferir el protocolo al proyecto aplicado.

La pregunta central es:

> ¿Qué resultado puedo creer cuando comparo modelos, hiperparámetros, métricas y thresholds?

## Tabla de contenidos

1. Contrato metodológico de la sesión  
2. Setup reproducible e instalación  
3. Funciones auxiliares maduras  
4. Dataset sintético desbalanceado y auditoría  
5. Split train/test y validación común  
6. Baselines  
7. KNN y SVM bajo el mismo protocolo  
8. Visualización de fronteras de decisión  
9. Curvas de validación: \(k\), \(C\), \(\gamma\)  
10. HPO formal con `RandomizedSearchCV`  
11. Imbalance handling: class weights y SMOTE seguro  
12. Threshold tuning con out-of-fold scores  
13. Evaluación final en test reservado  
14. Gaussian Processes como extensión conceptual  
15. Ejercicios y transferencia al proyecto  
16. Auditoría final, export y cierre

## 1. Contrato metodológico de la sesión

Durante todo el notebook se aplican estas reglas:

- El test set no se usa para seleccionar modelo, hiperparámetros, estrategia de imbalance ni threshold.
- El preprocessing que aprende de los datos vive dentro de un `Pipeline`.
- KNN y SVM se comparan bajo el mismo esquema de `StratifiedKFold`.
- La métrica principal será `average_precision`, equivalente operacional a PR-AUC en `scikit-learn`.
- Accuracy se reporta solo como referencia secundaria y nunca como métrica principal para imbalance.
- SMOTE, si se usa, debe estar dentro de un `imblearn.Pipeline`, nunca antes del split.
- El threshold se selecciona con out-of-fold scores, no con test.
- El test se reporta una sola vez al final.

> Un modelo no se defiende por entrenar bien, sino por generalizar bajo una validación honesta y tomar mejores decisiones bajo la métrica correcta.

In [ ]:
# =============================================================================
# 1.1 Methodological contract
# =============================================================================

methodological_contract = {
    "test_set_policy": "reserved_until_final_evaluation",
    "primary_metric": "average_precision / PR-AUC",
    "validation_protocol": "StratifiedKFold on training set",
    "preprocessing_policy": "learned preprocessing inside Pipeline",
    "hpo_policy": "validation_only_no_test",
    "threshold_policy": "out_of_fold_scores_no_test",
    "imbalance_policy": "metric_first_then_training_strategy",
    "smote_policy": "inside_imblearn_pipeline_only",
}

methodological_contract

## 2. Setup reproducible e instalación

Este notebook está diseñado para Colab, Kaggle y Jupyter local.

Librerías requeridas:

- `numpy`
- `pandas`
- `matplotlib`
- `scikit-learn`

Librería opcional:

- `imbalanced-learn`, solo si se desea ejecutar la estrategia con SMOTE.

Instalación opcional:

```bash
pip install -U scikit-learn imbalanced-learn
```

Después de instalar una librería en Colab o Jupyter, puede ser necesario reiniciar el runtime/kernel.

In [ ]:
# =============================================================================
# 2.1 Optional installation cell
# =============================================================================

# Uncomment only if needed.
# !pip install -U -q scikit-learn imbalanced-learn

In [ ]:
# =============================================================================
# 2.2 Imports and global configuration
# =============================================================================

from __future__ import annotations

import sys
import time
import warnings
from pathlib import Path
from typing import Any, Dict, List, Optional, Sequence

warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from IPython.display import display

import sklearn
from sklearn.base import BaseEstimator, clone
from sklearn.datasets import make_classification, make_moons
from sklearn.dummy import DummyClassifier
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import ConstantKernel, RBF, WhiteKernel
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    average_precision_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_recall_curve,
    precision_score,
    recall_score,
    roc_auc_score,
    roc_curve,
)
from sklearn.model_selection import (
    RandomizedSearchCV,
    StratifiedKFold,
    cross_val_predict,
    cross_validate,
    train_test_split,
)
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

try:
    from imblearn.pipeline import Pipeline as ImbPipeline
    from imblearn.over_sampling import SMOTE
    IMBLEARN_AVAILABLE = True
except Exception as exc:
    ImbPipeline = None
    SMOTE = None
    IMBLEARN_AVAILABLE = False
    print(f"imbalanced-learn no está disponible. SMOTE se omitirá. Detalle: {exc}")

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

FAST_DEMO_MODE = True
N_SPLITS = 5
HPO_N_SPLITS = 3 if FAST_DEMO_MODE else 5
TEST_SIZE = 0.20
N_JOBS = -1
POSITIVE_LABEL = 1
NEGATIVE_LABEL = 0

PRIMARY_SCORING = "average_precision"
RANDOM_SEARCH_N_ITER = 12 if FAST_DEMO_MODE else 32

THRESHOLD_POLICY = "max_f1"
TARGET_RECALL_FLOOR = 0.70
MAX_ALERT_RATE = 0.12

CLASSIFICATION_SCORING = {
    "pr_auc": "average_precision",
    "roc_auc": "roc_auc",
    "balanced_accuracy": "balanced_accuracy",
    "precision": "precision",
    "recall": "recall",
    "f1": "f1",
}

pd.set_option("display.max_columns", 160)
pd.set_option("display.width", 180)
pd.set_option("display.float_format", lambda x: f"{x:,.6f}")

plt.rcParams["figure.figsize"] = (9, 5.2)
plt.rcParams["font.size"] = 11
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.25
plt.rcParams["axes.spines.top"] = False
plt.rcParams["axes.spines.right"] = False

results_registry: Dict[str, Any] = {}
audit_registry: Dict[str, Any] = {}
model_registry: Dict[str, BaseEstimator] = {}

environment_report = pd.DataFrame(
    [
        {"package": "python", "version": sys.version.split()[0], "available": True},
        {"package": "numpy", "version": np.__version__, "available": True},
        {"package": "pandas", "version": pd.__version__, "available": True},
        {"package": "matplotlib", "version": "installed", "available": True},
        {"package": "scikit-learn", "version": sklearn.__version__, "available": True},
        {"package": "imbalanced-learn", "version": "optional", "available": IMBLEARN_AVAILABLE},
    ]
)

display(environment_report)

## 3. Funciones auxiliares

Esta sección concentra utilidades para:

- impresión de secciones;
- auditorías;
- evaluación por cross-validation;
- visualización de métricas;
- extracción de scores;
- threshold tuning;
- export y cierre.

La idea es evitar lógica duplicada y mantener el notebook como un producto de software legible.

In [ ]:
# =============================================================================
# 3.1 General utility functions
# =============================================================================

def print_section(title: str, width: int = 96) -> None:
    print("\n" + "=" * width)
    print(title.center(width))
    print("=" * width)


def require_columns(dataframe: pd.DataFrame, required_columns: Sequence[str], dataframe_name: str = "dataframe") -> None:
    missing = [col for col in required_columns if col not in dataframe.columns]
    if missing:
        raise ValueError(
            f"{dataframe_name} is missing required columns: {missing}. "
            f"Available columns: {list(dataframe.columns)}"
        )


def audit_required_objects(required_object_names: Sequence[str], scope: Dict[str, Any]) -> pd.DataFrame:
    return pd.DataFrame(
        [
            {
                "object": name,
                "exists": name in scope,
                "type": type(scope[name]).__name__ if name in scope else None,
            }
            for name in required_object_names
        ]
    )


def audit_metric_ranges(metric_values: Dict[str, float]) -> pd.DataFrame:
    rows = []
    for metric_name, value in metric_values.items():
        value = float(value)
        if metric_name.endswith("_std") or "std" in metric_name:
            in_range = value >= 0.0
        else:
            in_range = 0.0 <= value <= 1.0

        rows.append(
            {
                "metric": metric_name,
                "value": value,
                "is_finite": bool(np.isfinite(value)),
                "in_expected_range": bool(in_range),
                "passed": bool(np.isfinite(value) and in_range),
            }
        )
    return pd.DataFrame(rows)


def audit_pipeline_contains_step(estimator: Any, step_name: str) -> bool:
    return hasattr(estimator, "named_steps") and step_name in estimator.named_steps


def audit_smote_is_inside_pipeline(models: Dict[str, Any]) -> pd.DataFrame:
    rows = []
    for name, estimator in models.items():
        requires_smote_check = "smote" in name.lower()
        if requires_smote_check:
            passed = hasattr(estimator, "named_steps") and "smote" in estimator.named_steps
        else:
            passed = True
        rows.append(
            {
                "model": name,
                "requires_smote_check": requires_smote_check,
                "has_named_steps": hasattr(estimator, "named_steps"),
                "contains_smote_step": audit_pipeline_contains_step(estimator, "smote"),
                "passed": passed,
            }
        )
    return pd.DataFrame(rows)


def get_binary_class_balance(y_data: pd.Series | np.ndarray, split_name: str) -> pd.DataFrame:
    y_series = pd.Series(y_data)
    counts = y_series.value_counts().sort_index()
    total = int(len(y_series))
    rows = []
    for label in [NEGATIVE_LABEL, POSITIVE_LABEL]:
        count = int(counts.get(label, 0))
        rows.append(
            {
                "split": split_name,
                "class_label": label,
                "class_name": "positive" if label == POSITIVE_LABEL else "negative",
                "count": count,
                "rate": count / total,
                "rate_percent": 100 * count / total,
            }
        )
    return pd.DataFrame(rows)


def add_model_selection_columns(results: pd.DataFrame, metric_mean_col: str = "pr_auc_mean", metric_std_col: str = "pr_auc_std") -> pd.DataFrame:
    output = results.copy()
    output["relative_variability"] = output[metric_std_col] / output[metric_mean_col].replace(0, np.nan)
    output["rank_by_mean"] = output[metric_mean_col].rank(ascending=False, method="min")
    output["rank_by_stability"] = output["relative_variability"].rank(ascending=True, method="min")
    return output


def select_display_metric_columns(results: pd.DataFrame) -> pd.DataFrame:
    preferred_cols = [
        "model_group", "model",
        "pr_auc_mean", "pr_auc_std",
        "roc_auc_mean", "roc_auc_std",
        "balanced_accuracy_mean", "balanced_accuracy_std",
        "precision_mean", "precision_std",
        "recall_mean", "recall_std",
        "f1_mean", "f1_std",
        "relative_variability",
        "fit_time_mean", "score_time_mean", "total_time_sec",
    ]
    cols = [col for col in preferred_cols if col in results.columns]
    return results[cols].copy()


def combine_results(result_tables: Sequence[pd.DataFrame], sort_metric: str = "pr_auc_mean", ascending: bool = False) -> pd.DataFrame:
    valid_tables = [table for table in result_tables if isinstance(table, pd.DataFrame) and not table.empty]
    if not valid_tables:
        return pd.DataFrame()
    combined = pd.concat(valid_tables, ignore_index=True)
    return combined.sort_values(sort_metric, ascending=ascending).reset_index(drop=True)


print("General utility functions ready.")

In [ ]:
# =============================================================================
# 3.2 Model evaluation utilities
# =============================================================================

def summarize_cv_result(model_name: str, model_group: str, cv_result: Dict[str, np.ndarray], total_time_sec: float) -> pd.DataFrame:
    row: Dict[str, Any] = {
        "model_group": model_group,
        "model": model_name,
        "fit_time_mean": float(np.mean(cv_result["fit_time"])),
        "score_time_mean": float(np.mean(cv_result["score_time"])),
        "total_time_sec": float(total_time_sec),
    }

    for key, values in cv_result.items():
        if key.startswith("test_"):
            metric = key.replace("test_", "")
            row[f"{metric}_mean"] = float(np.mean(values))
            row[f"{metric}_std"] = float(np.std(values, ddof=1))

    return pd.DataFrame([row])


def evaluate_model_cv(
    estimator: BaseEstimator,
    model_name: str,
    model_group: str,
    X_data: pd.DataFrame,
    y_data: pd.Series,
    cv_strategy: StratifiedKFold,
    scoring: Dict[str, str],
    n_jobs: int = N_JOBS,
) -> pd.DataFrame:
    start_time = time.time()
    cv_result = cross_validate(
        estimator=estimator,
        X=X_data,
        y=y_data,
        cv=cv_strategy,
        scoring=scoring,
        n_jobs=n_jobs,
        return_train_score=False,
        error_score="raise",
    )
    total_time_sec = time.time() - start_time
    return summarize_cv_result(model_name, model_group, cv_result, total_time_sec)


def evaluate_model_collection_cv(
    models: Dict[str, BaseEstimator],
    model_group: str,
    X_data: pd.DataFrame,
    y_data: pd.Series,
    cv_strategy: StratifiedKFold,
    scoring: Dict[str, str],
    n_jobs: int = N_JOBS,
) -> pd.DataFrame:
    reports = []
    for model_name, estimator in models.items():
        print_section(f"Evaluating {model_name}")
        report = evaluate_model_cv(
            estimator=estimator,
            model_name=model_name,
            model_group=model_group,
            X_data=X_data,
            y_data=y_data,
            cv_strategy=cv_strategy,
            scoring=scoring,
            n_jobs=n_jobs,
        )
        reports.append(report)
        display(select_display_metric_columns(add_model_selection_columns(report)))
    combined = combine_results(reports, "pr_auc_mean", ascending=False)
    return add_model_selection_columns(combined)


def plot_cv_metric_comparison(
    results: pd.DataFrame,
    metric_mean_col: str,
    metric_std_col: str,
    title: str,
    top_n: Optional[int] = None,
) -> None:
    require_columns(results, ["model", metric_mean_col, metric_std_col], "results")
    plot_df = results.sort_values(metric_mean_col, ascending=True).copy()
    if top_n is not None:
        plot_df = plot_df.tail(top_n)

    fig, ax = plt.subplots(figsize=(10, max(4.5, 0.42 * len(plot_df))))
    ax.barh(plot_df["model"], plot_df[metric_mean_col], xerr=plot_df[metric_std_col], alpha=0.85)
    ax.set_title(title)
    ax.set_xlabel(f"{metric_mean_col} +/- {metric_std_col}")
    ax.set_ylabel("Modelo / pipeline")
    plt.show()


def plot_fold_class_distribution(X_data: pd.DataFrame, y_data: pd.Series, cv_strategy: StratifiedKFold, title: str) -> pd.DataFrame:
    rows = []
    for fold, (_, val_idx) in enumerate(cv_strategy.split(X_data, y_data), start=1):
        y_fold = y_data.iloc[val_idx]
        for label in [NEGATIVE_LABEL, POSITIVE_LABEL]:
            rows.append(
                {
                    "fold": fold,
                    "class_label": label,
                    "class_name": "positive" if label == POSITIVE_LABEL else "negative",
                    "count": int((y_fold == label).sum()),
                    "rate": float((y_fold == label).mean()),
                }
            )
    fold_df = pd.DataFrame(rows)

    pivot_counts = fold_df.pivot(index="fold", columns="class_name", values="count")
    pivot_counts.plot(kind="bar", figsize=(8, 4.8), alpha=0.85)
    plt.title(title)
    plt.xlabel("Fold")
    plt.ylabel("Número de observaciones")
    plt.legend(title="Clase")
    plt.show()

    return fold_df


print("Model evaluation utilities ready.")

In [ ]:
# =============================================================================
# 3.3 Score, threshold and search utilities
# =============================================================================

def get_fitted_model_scores(fitted_estimator: BaseEstimator, X_data: pd.DataFrame, prefer_predict_proba: bool = True) -> np.ndarray:
    if prefer_predict_proba and hasattr(fitted_estimator, "predict_proba"):
        scores = fitted_estimator.predict_proba(X_data)[:, 1]
    elif hasattr(fitted_estimator, "decision_function"):
        raw_scores = np.asarray(fitted_estimator.decision_function(X_data), dtype=float)
        denominator = raw_scores.max() - raw_scores.min()
        scores = np.zeros_like(raw_scores) if denominator == 0 else (raw_scores - raw_scores.min()) / denominator
    elif hasattr(fitted_estimator, "predict_proba"):
        scores = fitted_estimator.predict_proba(X_data)[:, 1]
    else:
        raise AttributeError("Estimator does not expose predict_proba or decision_function.")

    scores = np.asarray(scores, dtype=float)
    if np.isnan(scores).any():
        raise ValueError("Scores contain NaN values.")
    return scores


def get_out_of_fold_scores(
    estimator: BaseEstimator,
    X_data: pd.DataFrame,
    y_data: pd.Series,
    cv_strategy: StratifiedKFold,
    prefer_predict_proba: bool = True,
    n_jobs: int = N_JOBS,
) -> np.ndarray:
    method = "predict_proba" if prefer_predict_proba and hasattr(estimator, "predict_proba") else "decision_function"

    try:
        raw = cross_val_predict(estimator=estimator, X=X_data, y=y_data, cv=cv_strategy, method=method, n_jobs=n_jobs)
        if method == "predict_proba":
            scores = raw[:, 1]
        else:
            raw_scores = np.asarray(raw, dtype=float)
            denominator = raw_scores.max() - raw_scores.min()
            scores = np.zeros_like(raw_scores) if denominator == 0 else (raw_scores - raw_scores.min()) / denominator
    except Exception:
        raw = cross_val_predict(estimator=estimator, X=X_data, y=y_data, cv=cv_strategy, method="decision_function", n_jobs=n_jobs)
        raw_scores = np.asarray(raw, dtype=float)
        denominator = raw_scores.max() - raw_scores.min()
        scores = np.zeros_like(raw_scores) if denominator == 0 else (raw_scores - raw_scores.min()) / denominator

    scores = np.asarray(scores, dtype=float)
    if np.isnan(scores).any():
        raise ValueError("OOF scores contain NaN values.")
    return scores


def compute_ranking_metrics(y_true: Sequence[int], scores: Sequence[float]) -> Dict[str, float]:
    return {"pr_auc": float(average_precision_score(y_true, scores)), "roc_auc": float(roc_auc_score(y_true, scores))}


def compute_binary_policy_metrics(y_true: Sequence[int], scores: Sequence[float], threshold: float) -> Dict[str, Any]:
    y_true_array = np.asarray(y_true)
    y_pred = (np.asarray(scores, dtype=float) >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true_array, y_pred, labels=[NEGATIVE_LABEL, POSITIVE_LABEL]).ravel()

    return {
        "threshold": float(threshold),
        "precision": float(precision_score(y_true_array, y_pred, zero_division=0)),
        "recall": float(recall_score(y_true_array, y_pred, zero_division=0)),
        "f1": float(f1_score(y_true_array, y_pred, zero_division=0)),
        "balanced_accuracy": float(balanced_accuracy_score(y_true_array, y_pred)),
        "tn": int(tn),
        "fp": int(fp),
        "fn": int(fn),
        "tp": int(tp),
        "predicted_positive_count": int(y_pred.sum()),
        "predicted_positive_rate": float(y_pred.mean()),
    }


def build_threshold_sweep_table(y_true: Sequence[int], scores: Sequence[float], thresholds: Optional[np.ndarray] = None) -> pd.DataFrame:
    if thresholds is None:
        thresholds = np.unique(
            np.concatenate(
                [
                    np.linspace(0.001, 0.050, 40),
                    np.linspace(0.055, 0.500, 80),
                    np.linspace(0.510, 0.990, 50),
                ]
            )
        )
    return pd.DataFrame([compute_binary_policy_metrics(y_true, scores, float(threshold)) for threshold in thresholds])


def select_threshold_by_policy(
    threshold_table: pd.DataFrame,
    policy: str = "max_f1",
    target_recall_floor: float = 0.70,
    max_alert_rate: float = 0.12,
) -> pd.Series:
    require_columns(threshold_table, ["threshold", "precision", "recall", "f1", "predicted_positive_rate"], "threshold_table")

    if policy == "max_f1":
        return threshold_table.sort_values(["f1", "recall", "precision"], ascending=[False, False, False]).iloc[0]

    if policy == "max_precision_with_recall_floor":
        candidates = threshold_table[threshold_table["recall"] >= target_recall_floor]
        if candidates.empty:
            raise ValueError("No threshold satisfies the recall floor.")
        return candidates.sort_values(["precision", "f1", "threshold"], ascending=[False, False, False]).iloc[0]

    if policy == "max_recall_with_alert_cap":
        candidates = threshold_table[threshold_table["predicted_positive_rate"] <= max_alert_rate]
        if candidates.empty:
            raise ValueError("No threshold satisfies the alert-rate cap.")
        return candidates.sort_values(["recall", "precision", "f1"], ascending=[False, False, False]).iloc[0]

    raise ValueError(f"Unsupported threshold policy: {policy}")


def plot_pr_and_roc_curves(y_true: Sequence[int], scores: Sequence[float], title_prefix: str) -> None:
    precision, recall, _ = precision_recall_curve(y_true, scores)
    fpr, tpr, _ = roc_curve(y_true, scores)
    pr_auc = average_precision_score(y_true, scores)
    roc_auc = roc_auc_score(y_true, scores)
    prevalence = float(np.mean(np.asarray(y_true) == POSITIVE_LABEL))

    fig, ax = plt.subplots(figsize=(7.2, 5.4))
    ax.plot(recall, precision, linewidth=2, label=f"PR-AUC={pr_auc:.4f}")
    ax.axhline(prevalence, linestyle="--", linewidth=1.4, label=f"Prevalencia={prevalence:.4f}")
    ax.set_title(f"{title_prefix} — Curva Precision-Recall")
    ax.set_xlabel("Recall")
    ax.set_ylabel("Precision")
    ax.legend()
    plt.show()

    fig, ax = plt.subplots(figsize=(7.2, 5.4))
    ax.plot(fpr, tpr, linewidth=2, label=f"ROC-AUC={roc_auc:.4f}")
    ax.plot([0, 1], [0, 1], linestyle="--", linewidth=1.4, label="Referencia aleatoria")
    ax.set_title(f"{title_prefix} — Curva ROC")
    ax.set_xlabel("False Positive Rate")
    ax.set_ylabel("True Positive Rate")
    ax.legend()
    plt.show()


def plot_threshold_sweep(threshold_table: pd.DataFrame, selected_threshold: Optional[float], title_prefix: str) -> None:
    fig, ax = plt.subplots(figsize=(10, 5.4))
    ax.plot(threshold_table["threshold"], threshold_table["precision"], label="Precision", linewidth=2)
    ax.plot(threshold_table["threshold"], threshold_table["recall"], label="Recall", linewidth=2)
    ax.plot(threshold_table["threshold"], threshold_table["f1"], label="F1", linewidth=2)
    if selected_threshold is not None:
        ax.axvline(selected_threshold, linestyle="--", linewidth=2, label="Threshold seleccionado")
    ax.axvline(0.50, linestyle=":", linewidth=2, label="Threshold 0.5")
    ax.set_title(f"{title_prefix} — Métricas según threshold")
    ax.set_xlabel("Threshold")
    ax.set_ylabel("Valor")
    ax.legend()
    plt.show()

    fig, ax = plt.subplots(figsize=(10, 5.4))
    ax.plot(threshold_table["threshold"], threshold_table["predicted_positive_count"], linewidth=2)
    if selected_threshold is not None:
        ax.axvline(selected_threshold, linestyle="--", linewidth=2, label="Threshold seleccionado")
    ax.axvline(0.50, linestyle=":", linewidth=2, label="Threshold 0.5")
    ax.set_title(f"{title_prefix} — Volumen de alertas según threshold")
    ax.set_xlabel("Threshold")
    ax.set_ylabel("Número de alertas")
    ax.legend()
    plt.show()


def summarize_search_cv_results(search_object: RandomizedSearchCV, model_name: str, model_group: str, top_n: int = 10) -> pd.DataFrame:
    search_df = pd.DataFrame(search_object.cv_results_).copy()
    param_cols = [col for col in search_df.columns if col.startswith("param_")]
    cols = ["mean_test_score", "std_test_score", "rank_test_score"] + param_cols
    output = search_df[cols].sort_values("rank_test_score").head(top_n).reset_index(drop=True)
    output.insert(0, "model", model_name)
    output.insert(0, "model_group", model_group)
    return output


def extract_best_search_record(search_object: RandomizedSearchCV, model_name: str, model_group: str) -> pd.DataFrame:
    return pd.DataFrame(
        [
            {
                "model_group": model_group,
                "model": model_name,
                "best_score": float(search_object.best_score_),
                "best_params": dict(search_object.best_params_),
                "test_set_used": False,
            }
        ]
    )


def interpret_hyperparameter(parameter_name: str) -> str:
    cleaned = parameter_name.replace("model__", "")
    mapping = {
        "n_neighbors": "Controla cuántos vecinos votan en KNN; menor valor aumenta flexibilidad.",
        "weights": "Define si todos los vecinos pesan igual o si los más cercanos pesan más.",
        "p": "Controla la métrica Minkowski: p=1 Manhattan, p=2 Euclidiana.",
        "C": "Controla penalización de violaciones del margen en SVM.",
        "gamma": "Controla qué tan local es la influencia de los puntos en RBF.",
        "kernel": "Define la noción de similitud y la geometría de la frontera.",
    }
    return mapping.get(cleaned, "Hiperparámetro específico del modelo.")


def build_hyperparameter_interpretation_table(search_object: RandomizedSearchCV) -> pd.DataFrame:
    return pd.DataFrame(
        [
            {"parameter": param, "value": value, "interpretation": interpret_hyperparameter(param)}
            for param, value in search_object.best_params_.items()
        ]
    )


print("Score, threshold and search utilities ready.")

## 4. Dataset sintético desbalanceado

La clase permite trabajar con un caso binario desbalanceado. Para que el notebook sea descargable y ejecutable sin archivos externos, se construye un dataset sintético con:

- features numéricas;
- clase positiva rara;
- ruido controlado;
- separación parcial;
- suficientes observaciones para demostrar CV, HPO e imbalance.

Este dataset no pretende representar un dominio real. Su función es enseñar el protocolo.

In [ ]:
# =============================================================================
# 4.1 Synthetic imbalanced binary classification dataset
# =============================================================================

n_samples = 4_000 if FAST_DEMO_MODE else 10_000

X_array, y_array = make_classification(
    n_samples=n_samples,
    n_features=20,
    n_informative=8,
    n_redundant=4,
    n_repeated=0,
    n_classes=2,
    n_clusters_per_class=2,
    weights=[0.94, 0.06],
    class_sep=1.15,
    flip_y=0.025,
    random_state=RANDOM_STATE,
)

feature_names = [f"feature_{idx:02d}" for idx in range(X_array.shape[1])]
X = pd.DataFrame(X_array, columns=feature_names)
y = pd.Series(y_array, name="target").astype(int)

dataset_summary = pd.DataFrame(
    [
        {
            "dataset": "synthetic_imbalanced_binary_classification",
            "n_samples": X.shape[0],
            "n_features": X.shape[1],
            "positive_rate": float((y == POSITIVE_LABEL).mean()),
            "negative_rate": float((y == NEGATIVE_LABEL).mean()),
            "external_file_required": False,
            "purpose": "validación, model selection, KNN, SVM, imbalance y threshold",
        }
    ]
)

class_balance_table = get_binary_class_balance(y, "full_dataset")

results_registry["dataset_summary"] = dataset_summary
results_registry["class_balance_table"] = class_balance_table

display(dataset_summary)
display(class_balance_table)

fig, ax = plt.subplots(figsize=(7.5, 4.8))
plot_df = class_balance_table.copy()
ax.bar(plot_df["class_name"], plot_df["count"], alpha=0.85)
ax.set_yscale("log")
ax.set_title("Distribución de clases — escala logarítmica")
ax.set_xlabel("Clase")
ax.set_ylabel("Número de observaciones")
plt.show()

### Interpretación

La clase positiva es minoritaria. Por eso, un modelo que prediga siempre la clase negativa podría tener una accuracy aparentemente alta y, al mismo tiempo, ser inútil para el evento relevante.

Por esta razón, el notebook usa PR-AUC como métrica principal.

In [ ]:
# =============================================================================
# 4.2 Accuracy trap with a trivial all-negative classifier
# =============================================================================

majority_accuracy = float((y == NEGATIVE_LABEL).mean())
trivial_recall_positive = 0.0

accuracy_trap_table = pd.DataFrame(
    [
        {
            "classifier": "always_negative",
            "accuracy": majority_accuracy,
            "positive_recall": trivial_recall_positive,
            "pedagogical_reading": "Alta accuracy puede ocultar recall cero en la clase positiva.",
        }
    ]
)

results_registry["accuracy_trap_table"] = accuracy_trap_table

display(accuracy_trap_table)

## 5. Split train/test y validación común

Creamos un test set estratificado que queda reservado hasta el final.

Todas las decisiones de selección ocurren con `X_train` y `y_train`.

In [ ]:
# =============================================================================
# 5.1 Stratified train/test split
# =============================================================================

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=TEST_SIZE,
    stratify=y,
    random_state=RANDOM_STATE,
)

split_summary = pd.concat(
    [
        get_binary_class_balance(y_train, "train"),
        get_binary_class_balance(y_test, "test"),
    ],
    ignore_index=True,
)

results_registry["split_summary"] = split_summary

display(split_summary)

overlap = set(X_train.index).intersection(set(X_test.index))
if overlap:
    raise RuntimeError("Train/test index overlap detected.")

print(f"X_train shape: {X_train.shape}")
print(f"X_test shape: {X_test.shape}")

In [ ]:
# =============================================================================
# 5.2 Shared cross-validation strategy
# =============================================================================

cv = StratifiedKFold(
    n_splits=N_SPLITS,
    shuffle=True,
    random_state=RANDOM_STATE,
)

cv_report = pd.DataFrame(
    [
        {
            "cv_strategy": "StratifiedKFold",
            "n_splits": N_SPLITS,
            "shuffle": True,
            "random_state": RANDOM_STATE,
            "primary_metric": PRIMARY_SCORING,
        }
    ]
)

results_registry["cv_report"] = cv_report

display(cv_report)

fold_distribution_report = plot_fold_class_distribution(
    X_data=X_train,
    y_data=y_train,
    cv_strategy=cv,
    title="Distribución de clases en StratifiedKFold",
)

results_registry["fold_distribution_report"] = fold_distribution_report

display(fold_distribution_report.head(10))

## 6. Baselines

Antes de comparar KNN y SVM, necesitamos referencias:

1. `DummyClassifier`: baseline trivial.
2. `LogisticRegression`: baseline serio, simple y defendible.

Un modelo no lineal debe justificar su complejidad frente a un baseline serio.

In [ ]:
# =============================================================================
# 6.1 Baseline models
# =============================================================================

baseline_models = {
    "dummy_most_frequent": DummyClassifier(strategy="most_frequent"),
    "dummy_stratified": DummyClassifier(strategy="stratified", random_state=RANDOM_STATE),
    "logistic_regression_balanced": Pipeline(
        steps=[
            ("scaler", StandardScaler()),
            (
                "model",
                LogisticRegression(
                    class_weight="balanced",
                    max_iter=2_000,
                    random_state=RANDOM_STATE,
                ),
            ),
        ]
    ),
}

model_registry.update(baseline_models)

baseline_results = evaluate_model_collection_cv(
    models=baseline_models,
    model_group="baseline",
    X_data=X_train,
    y_data=y_train,
    cv_strategy=cv,
    scoring=CLASSIFICATION_SCORING,
    n_jobs=N_JOBS,
)

results_registry["baseline_results"] = baseline_results

display(select_display_metric_columns(baseline_results))

plot_cv_metric_comparison(
    baseline_results,
    "pr_auc_mean",
    "pr_auc_std",
    "Baselines — PR-AUC promedio +/- desviación estándar",
)

In [ ]:
# =============================================================================
# 6.2 Baseline interpretation table
# =============================================================================

baseline_interpretation_table = pd.DataFrame(
    [
        {
            "baseline": "dummy_most_frequent",
            "role": "Muestra la trampa de predecir siempre la clase mayoritaria.",
            "expected_behavior": "PR-AUC cercano a la prevalencia y recall positivo nulo a threshold fijo.",
        },
        {
            "baseline": "dummy_stratified",
            "role": "Referencia aleatoria que respeta proporciones.",
            "expected_behavior": "Puede alertar positivos por azar, pero no aprende estructura.",
        },
        {
            "baseline": "logistic_regression_balanced",
            "role": "Baseline serio lineal con class weights.",
            "expected_behavior": "Referencia razonable para decidir si KNN/SVM aportan no linealidad.",
        },
    ]
)

results_registry["baseline_interpretation_table"] = baseline_interpretation_table
display(baseline_interpretation_table)

## 7. KNN y SVM bajo el mismo protocolo

Ahora comparamos modelos centrales de la sesión:

- KNN con diferentes valores iniciales de \(k\);
- SVM lineal;
- SVM RBF.

Todos se encapsulan en `Pipeline` con `StandardScaler`.

Para KNN y SVM, scaling no es decoración: cambia la geometría del espacio.

In [ ]:
# =============================================================================
# 7.1 KNN and SVM candidate models
# =============================================================================

knn_svm_candidate_models = {
    "knn_k_5": Pipeline(
        steps=[
            ("scaler", StandardScaler()),
            ("model", KNeighborsClassifier(n_neighbors=5)),
        ]
    ),
    "knn_k_21": Pipeline(
        steps=[
            ("scaler", StandardScaler()),
            ("model", KNeighborsClassifier(n_neighbors=21)),
        ]
    ),
    "svm_linear_balanced": Pipeline(
        steps=[
            ("scaler", StandardScaler()),
            (
                "model",
                SVC(
                    kernel="linear",
                    C=1.0,
                    class_weight="balanced",
                    probability=False,
                    random_state=RANDOM_STATE,
                ),
            ),
        ]
    ),
    "svm_rbf_balanced": Pipeline(
        steps=[
            ("scaler", StandardScaler()),
            (
                "model",
                SVC(
                    kernel="rbf",
                    C=1.0,
                    gamma="scale",
                    class_weight="balanced",
                    probability=False,
                    random_state=RANDOM_STATE,
                ),
            ),
        ]
    ),
}

model_registry.update(knn_svm_candidate_models)

knn_svm_results = evaluate_model_collection_cv(
    models=knn_svm_candidate_models,
    model_group="knn_svm_initial",
    X_data=X_train,
    y_data=y_train,
    cv_strategy=cv,
    scoring=CLASSIFICATION_SCORING,
    n_jobs=N_JOBS,
)

results_registry["knn_svm_results"] = knn_svm_results

initial_model_comparison = combine_results(
    [baseline_results, knn_svm_results],
    sort_metric="pr_auc_mean",
    ascending=False,
)
initial_model_comparison = add_model_selection_columns(initial_model_comparison)

results_registry["initial_model_comparison"] = initial_model_comparison

display(select_display_metric_columns(initial_model_comparison))

plot_cv_metric_comparison(
    initial_model_comparison,
    "pr_auc_mean",
    "pr_auc_std",
    "Comparación inicial — PR-AUC promedio +/- desviación estándar",
    top_n=10,
)

### Interpretación

La comparación justa no es “KNN vs SVM” en abstracto.

La unidad real de comparación es:

> modelo + preprocessing + hiperparámetros + métrica + protocolo de validación

## 8. Visualización 2D de fronteras de decisión

La comparación principal usa un dataset tabular con 20 features.

Para enseñar geometría, usamos un dataset 2D sintético independiente. Su propósito es visualizar:

- KNN con \(k\) pequeño y grande;
- SVM lineal;
- SVM RBF.

Este bloque es conceptual y no participa en la selección final del modelo.

In [ ]:
# =============================================================================
# 8.1 Decision-boundary plotting utilities and 2D dataset
# =============================================================================

def plot_2d_data_and_decision_boundary(
    estimator: BaseEstimator,
    X_2d: np.ndarray,
    y_2d: np.ndarray,
    title: str,
    grid_resolution: int = 250,
) -> None:
    estimator = clone(estimator)
    estimator.fit(X_2d, y_2d)

    x_min, x_max = X_2d[:, 0].min() - 0.8, X_2d[:, 0].max() + 0.8
    y_min, y_max = X_2d[:, 1].min() - 0.8, X_2d[:, 1].max() + 0.8
    xx, yy = np.meshgrid(
        np.linspace(x_min, x_max, grid_resolution),
        np.linspace(y_min, y_max, grid_resolution),
    )
    grid = np.c_[xx.ravel(), yy.ravel()]
    zz = estimator.predict(grid).reshape(xx.shape)

    fig, ax = plt.subplots(figsize=(7, 5.8))
    ax.contourf(xx, yy, zz, alpha=0.35)
    ax.scatter(X_2d[:, 0], X_2d[:, 1], c=y_2d, s=24, edgecolor="black", alpha=0.85)
    ax.set_title(title)
    ax.set_xlabel("feature 1")
    ax.set_ylabel("feature 2")
    plt.show()


X_2d, y_2d = make_moons(
    n_samples=500,
    noise=0.28,
    random_state=RANDOM_STATE,
)

X_2d_train, X_2d_test, y_2d_train, y_2d_test = train_test_split(
    X_2d,
    y_2d,
    test_size=0.30,
    stratify=y_2d,
    random_state=RANDOM_STATE,
)

decision_boundary_models = {
    "KNN k=1": Pipeline([("scaler", StandardScaler()), ("model", KNeighborsClassifier(n_neighbors=1))]),
    "KNN k=35": Pipeline([("scaler", StandardScaler()), ("model", KNeighborsClassifier(n_neighbors=35))]),
    "SVM lineal": Pipeline([("scaler", StandardScaler()), ("model", SVC(kernel="linear", C=1.0, random_state=RANDOM_STATE))]),
    "SVM RBF": Pipeline([("scaler", StandardScaler()), ("model", SVC(kernel="rbf", C=3.0, gamma=0.8, random_state=RANDOM_STATE))]),
}

for boundary_name, boundary_model in decision_boundary_models.items():
    plot_2d_data_and_decision_boundary(
        estimator=boundary_model,
        X_2d=X_2d_train,
        y_2d=y_2d_train,
        title=f"Frontera de decisión — {boundary_name}",
    )

## 9. Curvas de validación para hiperparámetros clave

Antes de hacer HPO formal, observamos efectos individuales:

- \(k\) en KNN;
- \(C\) y \(\gamma\) en SVM RBF.

Estas curvas ayudan a interpretar HPO como una búsqueda de complejidad, no como magia.

In [ ]:
# =============================================================================
# 9.1 KNN validation curve over k
# =============================================================================

k_values = [1, 3, 5, 9, 15, 21, 31, 51, 75, 101]
knn_validation_rows = []

for k_value in k_values:
    estimator = Pipeline(
        steps=[
            ("scaler", StandardScaler()),
            ("model", KNeighborsClassifier(n_neighbors=k_value)),
        ]
    )
    report = evaluate_model_cv(
        estimator=estimator,
        model_name=f"knn_k_{k_value}",
        model_group="knn_validation_curve",
        X_data=X_train,
        y_data=y_train,
        cv_strategy=cv,
        scoring=CLASSIFICATION_SCORING,
        n_jobs=N_JOBS,
    )
    report["k"] = k_value
    knn_validation_rows.append(report)

knn_validation_curve_results = combine_results(
    knn_validation_rows,
    sort_metric="k",
    ascending=True,
)

results_registry["knn_validation_curve_results"] = knn_validation_curve_results

display(select_display_metric_columns(add_model_selection_columns(knn_validation_curve_results)))

fig, ax = plt.subplots(figsize=(9, 5.2))
ax.errorbar(
    knn_validation_curve_results["k"],
    knn_validation_curve_results["pr_auc_mean"],
    yerr=knn_validation_curve_results["pr_auc_std"],
    marker="o",
    linewidth=2,
    capsize=4,
)
ax.set_title("KNN — PR-AUC según número de vecinos k")
ax.set_xlabel("k vecinos")
ax.set_ylabel("PR-AUC promedio en CV")
plt.show()

In [ ]:
# =============================================================================
# 9.2 SVM RBF small C-gamma grid
# =============================================================================

svm_c_values = [0.1, 1.0, 10.0]
svm_gamma_values = [0.01, 0.1, 1.0]
svm_grid_rows = []

for c_value in svm_c_values:
    for gamma_value in svm_gamma_values:
        estimator = Pipeline(
            steps=[
                ("scaler", StandardScaler()),
                (
                    "model",
                    SVC(
                        kernel="rbf",
                        C=c_value,
                        gamma=gamma_value,
                        class_weight="balanced",
                        random_state=RANDOM_STATE,
                    ),
                ),
            ]
        )
        report = evaluate_model_cv(
            estimator=estimator,
            model_name=f"svm_rbf_C_{c_value}_gamma_{gamma_value}",
            model_group="svm_small_grid",
            X_data=X_train,
            y_data=y_train,
            cv_strategy=cv,
            scoring=CLASSIFICATION_SCORING,
            n_jobs=N_JOBS,
        )
        report["C"] = c_value
        report["gamma"] = gamma_value
        svm_grid_rows.append(report)

svm_small_grid_results = combine_results(
    svm_grid_rows,
    sort_metric="pr_auc_mean",
    ascending=False,
)

results_registry["svm_small_grid_results"] = svm_small_grid_results

display(select_display_metric_columns(add_model_selection_columns(svm_small_grid_results)))

heatmap_df = svm_small_grid_results.pivot(index="C", columns="gamma", values="pr_auc_mean")

fig, ax = plt.subplots(figsize=(7, 5.2))
im = ax.imshow(heatmap_df.values, aspect="auto")
ax.set_xticks(np.arange(len(heatmap_df.columns)))
ax.set_xticklabels(heatmap_df.columns)
ax.set_yticks(np.arange(len(heatmap_df.index)))
ax.set_yticklabels(heatmap_df.index)
ax.set_xlabel("gamma")
ax.set_ylabel("C")
ax.set_title("SVM RBF — PR-AUC promedio en grid C-gamma")
plt.colorbar(im, ax=ax, label="PR-AUC")
plt.show()

## 10. HPO formal con `RandomizedSearchCV`

HPO debe ocurrir dentro de validación. No reemplaza el test.

Optimizaremos:

- KNN: `n_neighbors`, `weights`, `p`;
- SVM RBF: `C`, `gamma`.

Luego reevaluaremos los mejores modelos bajo el CV común.

In [ ]:
# =============================================================================
# 10.1 HPO protocol and search spaces
# =============================================================================

hpo_cv = StratifiedKFold(
    n_splits=HPO_N_SPLITS,
    shuffle=True,
    random_state=RANDOM_STATE,
)

hpo_protocol_report = pd.DataFrame(
    [
        {
            "hpo_cv_strategy": "StratifiedKFold",
            "hpo_n_splits": HPO_N_SPLITS,
            "primary_scoring": PRIMARY_SCORING,
            "random_search_n_iter": RANDOM_SEARCH_N_ITER,
            "uses_training_set_only": True,
            "test_set_used": False,
        }
    ]
)

results_registry["hpo_protocol_report"] = hpo_protocol_report
display(hpo_protocol_report)

knn_hpo_pipeline = Pipeline(
    steps=[
        ("scaler", StandardScaler()),
        ("model", KNeighborsClassifier()),
    ]
)

knn_param_distributions = {
    "model__n_neighbors": [3, 5, 9, 15, 21, 31, 51, 75],
    "model__weights": ["uniform", "distance"],
    "model__p": [1, 2],
}

svm_rbf_hpo_pipeline = Pipeline(
    steps=[
        ("scaler", StandardScaler()),
        (
            "model",
            SVC(
                kernel="rbf",
                class_weight="balanced",
                probability=False,
                random_state=RANDOM_STATE,
            ),
        ),
    ]
)

svm_rbf_param_distributions = {
    "model__C": [0.05, 0.1, 0.3, 1.0, 3.0, 10.0, 30.0],
    "model__gamma": [0.001, 0.003, 0.01, 0.03, 0.1, 0.3, 1.0],
}

hpo_search_space_summary = pd.DataFrame(
    [
        {
            "search_name": "knn_random_search",
            "parameter": parameter,
            "values": str(values),
            "interpretation": interpret_hyperparameter(parameter),
        }
        for parameter, values in knn_param_distributions.items()
    ]
    + [
        {
            "search_name": "svm_rbf_random_search",
            "parameter": parameter,
            "values": str(values),
            "interpretation": interpret_hyperparameter(parameter),
        }
        for parameter, values in svm_rbf_param_distributions.items()
    ]
)

results_registry["hpo_search_space_summary"] = hpo_search_space_summary
display(hpo_search_space_summary)

In [ ]:
# =============================================================================
# 10.2 KNN RandomizedSearchCV
# =============================================================================

knn_search = RandomizedSearchCV(
    estimator=knn_hpo_pipeline,
    param_distributions=knn_param_distributions,
    n_iter=min(RANDOM_SEARCH_N_ITER, 16),
    scoring=PRIMARY_SCORING,
    cv=hpo_cv,
    random_state=RANDOM_STATE,
    n_jobs=N_JOBS,
    refit=True,
    verbose=1,
    error_score="raise",
)

knn_hpo_start_time = time.time()
knn_search.fit(X_train, y_train)
knn_hpo_elapsed_time = time.time() - knn_hpo_start_time

knn_hpo_results = summarize_search_cv_results(
    search_object=knn_search,
    model_name="knn_hpo_random_search",
    model_group="hpo_knn",
    top_n=10,
)

knn_hpo_best_record = extract_best_search_record(
    search_object=knn_search,
    model_name="knn_hpo_tuned",
    model_group="hpo_knn",
)

knn_best_params_interpretation = build_hyperparameter_interpretation_table(knn_search)

model_registry["knn_hpo_tuned"] = knn_search.best_estimator_
results_registry["knn_hpo_results"] = knn_hpo_results
results_registry["knn_hpo_best_record"] = knn_hpo_best_record
results_registry["knn_best_params_interpretation"] = knn_best_params_interpretation

display(knn_hpo_results)
display(knn_hpo_best_record)
display(knn_best_params_interpretation)

print(f"KNN HPO time: {knn_hpo_elapsed_time:.2f} seconds")
print(f"Best KNN score: {knn_search.best_score_:.6f}")

In [ ]:
# =============================================================================
# 10.3 SVM RBF RandomizedSearchCV
# =============================================================================

svm_rbf_search = RandomizedSearchCV(
    estimator=svm_rbf_hpo_pipeline,
    param_distributions=svm_rbf_param_distributions,
    n_iter=min(RANDOM_SEARCH_N_ITER, 18),
    scoring=PRIMARY_SCORING,
    cv=hpo_cv,
    random_state=RANDOM_STATE,
    n_jobs=N_JOBS,
    refit=True,
    verbose=1,
    error_score="raise",
)

svm_hpo_start_time = time.time()
svm_rbf_search.fit(X_train, y_train)
svm_hpo_elapsed_time = time.time() - svm_hpo_start_time

svm_rbf_hpo_results = summarize_search_cv_results(
    search_object=svm_rbf_search,
    model_name="svm_rbf_hpo_random_search",
    model_group="hpo_svm_rbf",
    top_n=10,
)

svm_rbf_hpo_best_record = extract_best_search_record(
    search_object=svm_rbf_search,
    model_name="svm_rbf_hpo_tuned",
    model_group="hpo_svm_rbf",
)

svm_rbf_best_params_interpretation = build_hyperparameter_interpretation_table(svm_rbf_search)

model_registry["svm_rbf_hpo_tuned"] = svm_rbf_search.best_estimator_
results_registry["svm_rbf_hpo_results"] = svm_rbf_hpo_results
results_registry["svm_rbf_hpo_best_record"] = svm_rbf_hpo_best_record
results_registry["svm_rbf_best_params_interpretation"] = svm_rbf_best_params_interpretation

display(svm_rbf_hpo_results)
display(svm_rbf_hpo_best_record)
display(svm_rbf_best_params_interpretation)

print(f"SVM HPO time: {svm_hpo_elapsed_time:.2f} seconds")
print(f"Best SVM score: {svm_rbf_search.best_score_:.6f}")

In [ ]:
# =============================================================================
# 10.4 Reevaluate tuned models under common CV
# =============================================================================

tuned_models = {
    "knn_hpo_tuned": knn_search.best_estimator_,
    "svm_rbf_hpo_tuned": svm_rbf_search.best_estimator_,
}

tuned_model_results = evaluate_model_collection_cv(
    models=tuned_models,
    model_group="hpo_tuned_common_cv",
    X_data=X_train,
    y_data=y_train,
    cv_strategy=cv,
    scoring=CLASSIFICATION_SCORING,
    n_jobs=N_JOBS,
)

results_registry["tuned_model_results"] = tuned_model_results

comparison_after_hpo = combine_results(
    [baseline_results, knn_svm_results, tuned_model_results],
    sort_metric="pr_auc_mean",
    ascending=False,
).drop_duplicates(subset=["model"], keep="last").reset_index(drop=True)

comparison_after_hpo = add_model_selection_columns(comparison_after_hpo)

results_registry["comparison_after_hpo"] = comparison_after_hpo

display(select_display_metric_columns(comparison_after_hpo))

plot_cv_metric_comparison(
    comparison_after_hpo,
    "pr_auc_mean",
    "pr_auc_std",
    "Comparación después de HPO — PR-AUC promedio +/- desviación estándar",
    top_n=12,
)

## 11. Imbalance handling: class weights y SMOTE seguro

Compararemos estrategias de entrenamiento para SVM RBF:

- sin `class_weight`;
- con `class_weight="balanced"`;
- con SMOTE dentro de `imblearn.Pipeline`, si la librería está disponible.

SMOTE antes del split o antes de cross-validation es leakage.

In [ ]:
# =============================================================================
# 11.1 Resampling leakage warning
# =============================================================================

resampling_leakage_warning = pd.DataFrame(
    [
        {
            "workflow": "SMOTE before train/test split",
            "status": "invalid",
            "why": "Synthetic points may leak structure from future validation/test data.",
        },
        {
            "workflow": "SMOTE before cross-validation",
            "status": "invalid",
            "why": "Validation folds may contain synthetic relatives of training points.",
        },
        {
            "workflow": "SMOTE inside imblearn Pipeline evaluated by CV",
            "status": "valid_if_appropriate",
            "why": "SMOTE is fitted only on the training portion of each fold.",
        },
        {
            "workflow": "Class weights inside estimator",
            "status": "valid_if_evaluated_correctly",
            "why": "Changes the training objective without synthesizing examples.",
        },
    ]
)

results_registry["resampling_leakage_warning"] = resampling_leakage_warning
display(resampling_leakage_warning)

In [ ]:
# =============================================================================
# 11.2 Define imbalance strategy models
# =============================================================================

svm_best_params = dict(svm_rbf_search.best_params_)
svm_best_c = float(svm_best_params.get("model__C", 1.0))
svm_best_gamma = svm_best_params.get("model__gamma", "scale")

imbalance_strategy_models: Dict[str, BaseEstimator] = {}

imbalance_strategy_models["svm_rbf_tuned_no_class_weight"] = Pipeline(
    steps=[
        ("scaler", StandardScaler()),
        (
            "model",
            SVC(
                kernel="rbf",
                C=svm_best_c,
                gamma=svm_best_gamma,
                class_weight=None,
                probability=False,
                random_state=RANDOM_STATE,
            ),
        ),
    ]
)

imbalance_strategy_models["svm_rbf_tuned_class_weight_balanced"] = Pipeline(
    steps=[
        ("scaler", StandardScaler()),
        (
            "model",
            SVC(
                kernel="rbf",
                C=svm_best_c,
                gamma=svm_best_gamma,
                class_weight="balanced",
                probability=False,
                random_state=RANDOM_STATE,
            ),
        ),
    ]
)

if IMBLEARN_AVAILABLE and ImbPipeline is not None and SMOTE is not None:
    imbalance_strategy_models["svm_rbf_tuned_smote_pipeline"] = ImbPipeline(
        steps=[
            ("scaler", StandardScaler()),
            ("smote", SMOTE(sampling_strategy=0.35, k_neighbors=5, random_state=RANDOM_STATE)),
            (
                "model",
                SVC(
                    kernel="rbf",
                    C=svm_best_c,
                    gamma=svm_best_gamma,
                    class_weight=None,
                    probability=False,
                    random_state=RANDOM_STATE,
                ),
            ),
        ]
    )
else:
    print("SMOTE strategy skipped because imbalanced-learn is unavailable.")

model_registry.update(imbalance_strategy_models)

imbalance_strategy_summary = pd.DataFrame(
    [
        {
            "model": model_name,
            "uses_pipeline": hasattr(estimator, "named_steps"),
            "contains_scaler_step": audit_pipeline_contains_step(estimator, "scaler"),
            "contains_smote_step": audit_pipeline_contains_step(estimator, "smote"),
            "uses_class_weight": "class_weight_balanced" in model_name,
            "test_set_used": False,
        }
        for model_name, estimator in imbalance_strategy_models.items()
    ]
)

results_registry["imbalance_strategy_summary"] = imbalance_strategy_summary
display(imbalance_strategy_summary)

In [ ]:
# =============================================================================
# 11.3 Evaluate imbalance strategies
# =============================================================================

imbalance_strategy_results = evaluate_model_collection_cv(
    models=imbalance_strategy_models,
    model_group="imbalance_strategy",
    X_data=X_train,
    y_data=y_train,
    cv_strategy=cv,
    scoring=CLASSIFICATION_SCORING,
    n_jobs=N_JOBS,
)

results_registry["imbalance_strategy_results"] = imbalance_strategy_results

comparison_with_imbalance = combine_results(
    [baseline_results, knn_svm_results, tuned_model_results, imbalance_strategy_results],
    sort_metric="pr_auc_mean",
    ascending=False,
).drop_duplicates(subset=["model"], keep="last").reset_index(drop=True)

comparison_with_imbalance = add_model_selection_columns(comparison_with_imbalance)

results_registry["comparison_with_imbalance"] = comparison_with_imbalance

display(select_display_metric_columns(comparison_with_imbalance))

plot_cv_metric_comparison(
    comparison_with_imbalance,
    "pr_auc_mean",
    "pr_auc_std",
    "Comparación final pre-threshold — PR-AUC promedio +/- desviación estándar",
    top_n=14,
)

In [ ]:
# =============================================================================
# 11.4 Select final candidate before threshold tuning
# =============================================================================

candidate_cv_row = comparison_with_imbalance.sort_values(
    ["pr_auc_mean", "recall_mean", "f1_mean"],
    ascending=[False, False, False],
).iloc[0]

CANDIDATE_MODEL_NAME = str(candidate_cv_row["model"])
CANDIDATE_MODEL_GROUP = str(candidate_cv_row["model_group"])
CANDIDATE_PR_AUC_MEAN = float(candidate_cv_row["pr_auc_mean"])
CANDIDATE_PR_AUC_STD = float(candidate_cv_row["pr_auc_std"])
CANDIDATE_RECALL_MEAN = float(candidate_cv_row["recall_mean"])
CANDIDATE_PRECISION_MEAN = float(candidate_cv_row["precision_mean"])
CANDIDATE_F1_MEAN = float(candidate_cv_row["f1_mean"])

if CANDIDATE_MODEL_NAME not in model_registry:
    raise ValueError(f"Candidate model not found: {CANDIDATE_MODEL_NAME}")

selected_model = model_registry[CANDIDATE_MODEL_NAME]

candidate_selection_record = pd.DataFrame(
    [
        {
            "candidate_model": CANDIDATE_MODEL_NAME,
            "candidate_group": CANDIDATE_MODEL_GROUP,
            "selection_metric": PRIMARY_SCORING,
            "cv_pr_auc_mean": CANDIDATE_PR_AUC_MEAN,
            "cv_pr_auc_std": CANDIDATE_PR_AUC_STD,
            "cv_recall_mean": CANDIDATE_RECALL_MEAN,
            "cv_precision_mean": CANDIDATE_PRECISION_MEAN,
            "cv_f1_mean": CANDIDATE_F1_MEAN,
            "test_set_used": False,
        }
    ]
)

results_registry["candidate_selection_record"] = candidate_selection_record

display(candidate_selection_record)
print(selected_model)

## 12. Threshold tuning con out-of-fold scores

El modelo ya fue seleccionado por CV, pero falta transformar scores en decisiones.

El threshold se selecciona con out-of-fold scores sobre training, no con test.

In [ ]:
# =============================================================================
# 12.1 Out-of-fold scores
# =============================================================================

print_section(f"OOF scores for selected candidate: {CANDIDATE_MODEL_NAME}")

oof_scores = get_out_of_fold_scores(
    estimator=selected_model,
    X_data=X_train,
    y_data=y_train,
    cv_strategy=cv,
    prefer_predict_proba=True,
    n_jobs=N_JOBS,
)

oof_ranking_metrics = compute_ranking_metrics(y_train, oof_scores)

OOF_PR_AUC = float(oof_ranking_metrics["pr_auc"])
OOF_ROC_AUC = float(oof_ranking_metrics["roc_auc"])

oof_score_summary = pd.DataFrame(
    [
        {
            "candidate_model": CANDIDATE_MODEL_NAME,
            "score_count": len(oof_scores),
            "min_score": float(np.min(oof_scores)),
            "median_score": float(np.median(oof_scores)),
            "max_score": float(np.max(oof_scores)),
            "oof_pr_auc": OOF_PR_AUC,
            "oof_roc_auc": OOF_ROC_AUC,
            "test_set_used": False,
        }
    ]
)

results_registry["oof_score_summary"] = oof_score_summary
display(oof_score_summary)

plot_pr_and_roc_curves(
    y_true=y_train,
    scores=oof_scores,
    title_prefix=f"OOF — {CANDIDATE_MODEL_NAME}",
)

In [ ]:
# =============================================================================
# 12.2 Score distribution by class
# =============================================================================

oof_score_df = pd.DataFrame(
    {
        "true_label": y_train.to_numpy(),
        "score": oof_scores,
    }
)

results_registry["oof_score_df"] = oof_score_df

display(oof_score_df.groupby("true_label")["score"].describe())

fig, ax = plt.subplots(figsize=(9, 5.4))
ax.hist(
    oof_score_df.query("true_label == @NEGATIVE_LABEL")["score"],
    bins=50,
    alpha=0.70,
    label="Clase negativa",
)
ax.hist(
    oof_score_df.query("true_label == @POSITIVE_LABEL")["score"],
    bins=50,
    alpha=0.70,
    label="Clase positiva",
)
ax.set_yscale("log")
ax.set_title("Distribución de scores OOF por clase")
ax.set_xlabel("Score")
ax.set_ylabel("Frecuencia, escala logarítmica")
ax.legend()
plt.show()

In [ ]:
# =============================================================================
# 12.3 Threshold sweep and policy selection
# =============================================================================

threshold_sweep_df = build_threshold_sweep_table(
    y_true=y_train,
    scores=oof_scores,
)

selected_threshold_row = select_threshold_by_policy(
    threshold_table=threshold_sweep_df,
    policy=THRESHOLD_POLICY,
    target_recall_floor=TARGET_RECALL_FLOOR,
    max_alert_rate=MAX_ALERT_RATE,
)

SELECTED_THRESHOLD = float(selected_threshold_row["threshold"])

selected_threshold_metrics = compute_binary_policy_metrics(
    y_true=y_train,
    scores=oof_scores,
    threshold=SELECTED_THRESHOLD,
)

default_threshold_metrics = compute_binary_policy_metrics(
    y_true=y_train,
    scores=oof_scores,
    threshold=0.50,
)

threshold_policy_comparison = pd.DataFrame(
    [
        {"policy": "default_0.50", **default_threshold_metrics},
        {"policy": THRESHOLD_POLICY, **selected_threshold_metrics},
    ]
)

threshold_policy_comparison.insert(0, "candidate_model", CANDIDATE_MODEL_NAME)

results_registry["threshold_sweep_df"] = threshold_sweep_df
results_registry["threshold_policy_comparison"] = threshold_policy_comparison

display(threshold_sweep_df.sort_values(["f1", "recall", "precision"], ascending=[False, False, False]).head(20))
display(threshold_policy_comparison)

plot_threshold_sweep(
    threshold_table=threshold_sweep_df,
    selected_threshold=SELECTED_THRESHOLD,
    title_prefix=f"OOF — {CANDIDATE_MODEL_NAME}",
)

print(f"Selected threshold: {SELECTED_THRESHOLD:.6f}")

In [ ]:
# =============================================================================
# 12.4 OOF confusion matrices
# =============================================================================

def build_confusion_matrix_dataframe(y_true: Sequence[int], scores: Sequence[float], threshold: float) -> pd.DataFrame:
    y_pred = (np.asarray(scores, dtype=float) >= threshold).astype(int)
    cm = confusion_matrix(y_true, y_pred, labels=[NEGATIVE_LABEL, POSITIVE_LABEL])
    return pd.DataFrame(
        cm,
        index=["actual_negative", "actual_positive"],
        columns=["predicted_negative", "predicted_positive"],
    )


oof_default_cm_df = build_confusion_matrix_dataframe(y_train, oof_scores, 0.50)
oof_selected_cm_df = build_confusion_matrix_dataframe(y_train, oof_scores, SELECTED_THRESHOLD)

results_registry["oof_default_cm_df"] = oof_default_cm_df
results_registry["oof_selected_cm_df"] = oof_selected_cm_df

print_section("OOF confusion matrix — default threshold 0.50")
display(oof_default_cm_df)

print_section(f"OOF confusion matrix — selected threshold {SELECTED_THRESHOLD:.6f}")
display(oof_selected_cm_df)

In [ ]:
# =============================================================================
# 12.5 Pre-test decision record
# =============================================================================

pretest_decision_record = pd.DataFrame(
    [
        {
            "stage": "before_final_test_evaluation",
            "candidate_model": CANDIDATE_MODEL_NAME,
            "candidate_group": CANDIDATE_MODEL_GROUP,
            "primary_metric": PRIMARY_SCORING,
            "cv_strategy": "StratifiedKFold",
            "cv_pr_auc_mean": CANDIDATE_PR_AUC_MEAN,
            "cv_pr_auc_std": CANDIDATE_PR_AUC_STD,
            "oof_pr_auc": OOF_PR_AUC,
            "oof_roc_auc": OOF_ROC_AUC,
            "threshold_policy": THRESHOLD_POLICY,
            "selected_threshold": SELECTED_THRESHOLD,
            "oof_precision_at_threshold": selected_threshold_metrics["precision"],
            "oof_recall_at_threshold": selected_threshold_metrics["recall"],
            "oof_f1_at_threshold": selected_threshold_metrics["f1"],
            "oof_alert_count": selected_threshold_metrics["predicted_positive_count"],
            "oof_alert_rate": selected_threshold_metrics["predicted_positive_rate"],
            "test_set_used_for_training": False,
            "test_set_used_for_hpo": False,
            "test_set_used_for_model_selection": False,
            "test_set_used_for_threshold_selection": False,
            "test_set_used": False,
        }
    ]
)

results_registry["pretest_decision_record"] = pretest_decision_record
display(pretest_decision_record)

## 13. Evaluación final en test reservado

Ahora sí se usa el test set.

Las decisiones están congeladas:

- candidato;
- hiperparámetros;
- estrategia de imbalance;
- threshold.

El test no debe usarse para reajustar nada.

In [ ]:
# =============================================================================
# 13.1 Final fit on full training set
# =============================================================================

final_model = clone(selected_model)

final_fit_start_time = time.time()
final_model.fit(X_train, y_train)
final_fit_time_sec = time.time() - final_fit_start_time

final_refit_record = pd.DataFrame(
    [
        {
            "final_model": CANDIDATE_MODEL_NAME,
            "fit_data": "X_train, y_train",
            "n_train_rows": len(X_train),
            "n_train_features": X_train.shape[1],
            "selected_threshold": SELECTED_THRESHOLD,
            "threshold_source": "out-of-fold training scores",
            "test_set_used_for_refit": False,
            "final_fit_time_sec": final_fit_time_sec,
        }
    ]
)

results_registry["final_refit_record"] = final_refit_record
display(final_refit_record)

In [ ]:
# =============================================================================
# 13.2 Final test scores and ranking metrics
# =============================================================================

y_test_scores = get_fitted_model_scores(
    fitted_estimator=final_model,
    X_data=X_test,
    prefer_predict_proba=True,
)

test_ranking_metrics = compute_ranking_metrics(
    y_true=y_test,
    scores=y_test_scores,
)

TEST_PR_AUC = float(test_ranking_metrics["pr_auc"])
TEST_ROC_AUC = float(test_ranking_metrics["roc_auc"])

test_ranking_record = pd.DataFrame(
    [
        {
            "final_model": CANDIDATE_MODEL_NAME,
            "test_pr_auc": TEST_PR_AUC,
            "test_roc_auc": TEST_ROC_AUC,
            "primary_metric": PRIMARY_SCORING,
            "threshold_applied": False,
            "test_set_used_once_for_final_evaluation": True,
        }
    ]
)

results_registry["test_ranking_record"] = test_ranking_record

display(test_ranking_record)

plot_pr_and_roc_curves(
    y_true=y_test,
    scores=y_test_scores,
    title_prefix=f"TEST — {CANDIDATE_MODEL_NAME}",
)

In [ ]:
# =============================================================================
# 13.3 Apply frozen threshold and evaluate final decisions
# =============================================================================

test_default_threshold_metrics = compute_binary_policy_metrics(
    y_true=y_test,
    scores=y_test_scores,
    threshold=0.50,
)

test_selected_threshold_metrics = compute_binary_policy_metrics(
    y_true=y_test,
    scores=y_test_scores,
    threshold=SELECTED_THRESHOLD,
)

test_threshold_comparison = pd.DataFrame(
    [
        {"policy": "default_0.50", **test_default_threshold_metrics},
        {"policy": THRESHOLD_POLICY, **test_selected_threshold_metrics},
    ]
)

test_threshold_comparison.insert(0, "final_model", CANDIDATE_MODEL_NAME)

results_registry["test_threshold_comparison"] = test_threshold_comparison

display(test_threshold_comparison)

y_test_pred_selected = (y_test_scores >= SELECTED_THRESHOLD).astype(int)

test_cm_selected = confusion_matrix(
    y_test,
    y_test_pred_selected,
    labels=[NEGATIVE_LABEL, POSITIVE_LABEL],
)

test_cm_selected_df = pd.DataFrame(
    test_cm_selected,
    index=["actual_negative", "actual_positive"],
    columns=["predicted_negative", "predicted_positive"],
)

results_registry["test_cm_selected_df"] = test_cm_selected_df

display(test_cm_selected_df)

fig, ax = plt.subplots(figsize=(6, 5))
ConfusionMatrixDisplay(
    confusion_matrix=test_cm_selected,
    display_labels=["Clase negativa", "Clase positiva"],
).plot(ax=ax, values_format="d")
ax.set_title(f"Matriz de confusión en test — threshold={SELECTED_THRESHOLD:.4f}")
plt.show()

print_section("Classification report — reserved test set")
print(
    classification_report(
        y_test,
        y_test_pred_selected,
        target_names=["negative_class", "positive_class"],
        zero_division=0,
    )
)

In [ ]:
# =============================================================================
# 13.4 Final evidence table and conclusion
# =============================================================================

final_evidence_table = pd.DataFrame(
    [
        {
            "stage": "cross_validation_selection",
            "model": CANDIDATE_MODEL_NAME,
            "data_source": "training folds",
            "pr_auc": CANDIDATE_PR_AUC_MEAN,
            "pr_auc_std": CANDIDATE_PR_AUC_STD,
            "roc_auc": np.nan,
            "threshold": np.nan,
            "precision": CANDIDATE_PRECISION_MEAN,
            "recall": CANDIDATE_RECALL_MEAN,
            "f1": CANDIDATE_F1_MEAN,
            "balanced_accuracy": np.nan,
            "alert_count": np.nan,
            "alert_rate": np.nan,
            "test_set_used_for_decision": False,
        },
        {
            "stage": "out_of_fold_threshold_design",
            "model": CANDIDATE_MODEL_NAME,
            "data_source": "out-of-fold training scores",
            "pr_auc": OOF_PR_AUC,
            "pr_auc_std": np.nan,
            "roc_auc": OOF_ROC_AUC,
            "threshold": SELECTED_THRESHOLD,
            "precision": selected_threshold_metrics["precision"],
            "recall": selected_threshold_metrics["recall"],
            "f1": selected_threshold_metrics["f1"],
            "balanced_accuracy": selected_threshold_metrics["balanced_accuracy"],
            "alert_count": selected_threshold_metrics["predicted_positive_count"],
            "alert_rate": selected_threshold_metrics["predicted_positive_rate"],
            "test_set_used_for_decision": False,
        },
        {
            "stage": "reserved_test_final_evaluation",
            "model": CANDIDATE_MODEL_NAME,
            "data_source": "reserved test set",
            "pr_auc": TEST_PR_AUC,
            "pr_auc_std": np.nan,
            "roc_auc": TEST_ROC_AUC,
            "threshold": SELECTED_THRESHOLD,
            "precision": test_selected_threshold_metrics["precision"],
            "recall": test_selected_threshold_metrics["recall"],
            "f1": test_selected_threshold_metrics["f1"],
            "balanced_accuracy": test_selected_threshold_metrics["balanced_accuracy"],
            "alert_count": test_selected_threshold_metrics["predicted_positive_count"],
            "alert_rate": test_selected_threshold_metrics["predicted_positive_rate"],
            "test_set_used_for_decision": False,
        },
    ]
)

final_experiment_record = pd.DataFrame(
    [
        {
            "course": "Aprendizaje de Máquina Aplicado — Electiva",
            "session": "Chapter 4 / Session 4",
            "session_title": "Validación, selección de modelos, KNN, SVM e imbalance",
            "dataset_type": "synthetic imbalanced binary classification",
            "final_model": CANDIDATE_MODEL_NAME,
            "model_group": CANDIDATE_MODEL_GROUP,
            "primary_metric": PRIMARY_SCORING,
            "selection_protocol": "StratifiedKFold on training set",
            "hpo_protocol": "RandomizedSearchCV on training set",
            "threshold_source": "out-of-fold training scores",
            "threshold_policy": THRESHOLD_POLICY,
            "selected_threshold": SELECTED_THRESHOLD,
            "cv_pr_auc_mean": CANDIDATE_PR_AUC_MEAN,
            "cv_pr_auc_std": CANDIDATE_PR_AUC_STD,
            "oof_pr_auc": OOF_PR_AUC,
            "oof_roc_auc": OOF_ROC_AUC,
            "test_pr_auc": TEST_PR_AUC,
            "test_roc_auc": TEST_ROC_AUC,
            "test_precision": test_selected_threshold_metrics["precision"],
            "test_recall": test_selected_threshold_metrics["recall"],
            "test_f1": test_selected_threshold_metrics["f1"],
            "test_balanced_accuracy": test_selected_threshold_metrics["balanced_accuracy"],
            "test_alert_count": test_selected_threshold_metrics["predicted_positive_count"],
            "test_alert_rate": test_selected_threshold_metrics["predicted_positive_rate"],
            "final_fit_time_sec": final_fit_time_sec,
            "test_set_used_for_training": False,
            "test_set_used_for_hpo": False,
            "test_set_used_for_model_selection": False,
            "test_set_used_for_threshold_selection": False,
            "test_set_used_once_for_final_evaluation": True,
        }
    ]
)

results_registry["final_evidence_table"] = final_evidence_table
results_registry["final_experiment_record"] = final_experiment_record

display(final_evidence_table)
display(final_experiment_record)

final_conclusion = f"""
Conclusión técnica final:

Bajo StratifiedKFold en el conjunto de entrenamiento y usando average_precision / PR-AUC
como métrica principal, el candidato seleccionado fue:

    {CANDIDATE_MODEL_NAME}

Evidencia de selección:
    CV PR-AUC promedio = {CANDIDATE_PR_AUC_MEAN:.6f}
    CV PR-AUC std      = {CANDIDATE_PR_AUC_STD:.6f}

Diseño de threshold:
    OOF PR-AUC         = {OOF_PR_AUC:.6f}
    OOF ROC-AUC        = {OOF_ROC_AUC:.6f}
    Política threshold = {THRESHOLD_POLICY}
    Threshold elegido  = {SELECTED_THRESHOLD:.6f}

Evaluación final en test reservado:
    Test PR-AUC        = {TEST_PR_AUC:.6f}
    Test ROC-AUC       = {TEST_ROC_AUC:.6f}
    Test precision     = {test_selected_threshold_metrics['precision']:.6f}
    Test recall        = {test_selected_threshold_metrics['recall']:.6f}
    Test F1            = {test_selected_threshold_metrics['f1']:.6f}
    Test balanced acc. = {test_selected_threshold_metrics['balanced_accuracy']:.6f}
    Test tasa alertas  = {test_selected_threshold_metrics['predicted_positive_rate']:.6f}

Lectura metodológica:
    La conclusión es defendible porque el modelo, los hiperparámetros, la estrategia
    de imbalance y el threshold se seleccionaron antes de mirar el test set.
"""

results_registry["final_conclusion"] = final_conclusion
print(final_conclusion)

## 14. Gaussian Processes como extensión conceptual

Esta sección no compite con KNN ni SVM en el flujo principal.

Su propósito es mostrar que algunos modelos no solo entregan predicción puntual, sino incertidumbre:

\[
f(x) \sim GP(m(x), k(x,x'))
\]

Usamos una regresión 1D pequeña para visualizar media predictiva, bandas de incertidumbre y funciones plausibles.

In [ ]:
# =============================================================================
# 14.1 Gaussian Process 1D regression demo
# =============================================================================

rng = np.random.default_rng(RANDOM_STATE)

X_gp_train = np.sort(rng.uniform(0.0, 10.0, size=28)).reshape(-1, 1)
true_function_gp = np.sin(X_gp_train[:, 0]) + 0.20 * X_gp_train[:, 0]
y_gp_train = true_function_gp + rng.normal(loc=0.0, scale=0.35, size=X_gp_train.shape[0])

X_gp_grid = np.linspace(-1.0, 11.0, 350).reshape(-1, 1)

gp_kernel = (
    ConstantKernel(1.0, constant_value_bounds=(0.1, 10.0))
    * RBF(length_scale=1.0, length_scale_bounds=(0.1, 10.0))
    + WhiteKernel(noise_level=0.1, noise_level_bounds=(1e-3, 1.0))
)

gp_model = GaussianProcessRegressor(
    kernel=gp_kernel,
    alpha=0.0,
    normalize_y=True,
    n_restarts_optimizer=5,
    random_state=RANDOM_STATE,
)

gp_model.fit(X_gp_train, y_gp_train)

gp_mean, gp_std = gp_model.predict(X_gp_grid, return_std=True)

true_function_grid = np.sin(X_gp_grid[:, 0]) + 0.20 * X_gp_grid[:, 0]
gp_lower_band = gp_mean - 1.96 * gp_std
gp_upper_band = gp_mean + 1.96 * gp_std

gp_model_summary = pd.DataFrame(
    [
        {
            "model": "GaussianProcessRegressor",
            "kernel_after_fit": str(gp_model.kernel_),
            "used_for_main_model_selection": False,
            "classification_test_set_used": False,
        }
    ]
)

results_registry["gp_model_summary"] = gp_model_summary
display(gp_model_summary)

fig, ax = plt.subplots(figsize=(10, 5.8))
ax.plot(X_gp_grid[:, 0], true_function_grid, linestyle="--", linewidth=2, label="Función verdadera sintética")
ax.plot(X_gp_grid[:, 0], gp_mean, linewidth=2.5, label="Media predictiva GP")
ax.fill_between(X_gp_grid[:, 0], gp_lower_band, gp_upper_band, alpha=0.25, label="Banda aproximada 95%")
ax.scatter(X_gp_train[:, 0], y_gp_train, s=45, edgecolor="black", label="Observaciones")
ax.set_title("Gaussian Process: predicción con incertidumbre")
ax.set_xlabel("x")
ax.set_ylabel("y")
ax.legend()
plt.show()

In [ ]:
# =============================================================================
# 14.2 Plausible functions sampled from the GP posterior
# =============================================================================

gp_function_samples = gp_model.sample_y(
    X_gp_grid,
    n_samples=6,
    random_state=RANDOM_STATE,
)

fig, ax = plt.subplots(figsize=(10, 5.8))
for sample_idx in range(gp_function_samples.shape[1]):
    ax.plot(
        X_gp_grid[:, 0],
        gp_function_samples[:, sample_idx],
        linewidth=1.4,
        alpha=0.75,
    )

ax.scatter(X_gp_train[:, 0], y_gp_train, s=45, edgecolor="black", label="Observaciones")
ax.plot(X_gp_grid[:, 0], gp_mean, linewidth=2.5, label="Media predictiva")
ax.set_title("Gaussian Process: funciones plausibles posteriores")
ax.set_xlabel("x")
ax.set_ylabel("y")
ax.legend()
plt.show()

knn_svm_gp_comparison = pd.DataFrame(
    [
        {
            "model_family": "KNN",
            "core_question": "¿Qué dicen los vecinos más cercanos?",
            "main_strength": "Intuición local simple.",
            "main_risk": "La distancia puede no representar similitud real.",
            "key_hyperparameters": "k, distancia, weights",
            "uncertainty_native": False,
        },
        {
            "model_family": "SVM",
            "core_question": "¿Qué frontera separa con margen defendible?",
            "main_strength": "Control geométrico mediante margen, C y kernel.",
            "main_risk": "Frontera compleja no implica mejor generalización.",
            "key_hyperparameters": "C, gamma, kernel",
            "uncertainty_native": False,
        },
        {
            "model_family": "Gaussian Process",
            "core_question": "¿Qué funciones son plausibles y con cuánta incertidumbre?",
            "main_strength": "Predicción con incertidumbre explícita.",
            "main_risk": "Costo computacional y dependencia del kernel.",
            "key_hyperparameters": "kernel, length_scale, noise_level",
            "uncertainty_native": True,
        },
    ]
)

results_registry["knn_svm_gp_comparison"] = knn_svm_gp_comparison
display(knn_svm_gp_comparison)

In [ ]:
# =============================================================================
# Final student-facing summary
# =============================================================================

student_facing_summary = f"""
Resumen final del notebook:

Modelo final:
    {CANDIDATE_MODEL_NAME}

Métrica principal:
    {PRIMARY_SCORING}

Protocolo:
    - Split estratificado con test reservado.
    - Comparación por StratifiedKFold.
    - HPO dentro de validación.
    - Threshold seleccionado con out-of-fold scores.
    - Test usado una sola vez al final.

Threshold seleccionado:
    {SELECTED_THRESHOLD:.6f}

Resultados finales en test reservado:
    PR-AUC             = {TEST_PR_AUC:.6f}
    ROC-AUC            = {TEST_ROC_AUC:.6f}
    Precision          = {test_selected_threshold_metrics['precision']:.6f}
    Recall             = {test_selected_threshold_metrics['recall']:.6f}
    F1                 = {test_selected_threshold_metrics['f1']:.6f}
    Balanced accuracy  = {test_selected_threshold_metrics['balanced_accuracy']:.6f}
    Tasa de alertas    = {test_selected_threshold_metrics['predicted_positive_rate']:.6f}

Mensaje metodológico:
    El resultado es defendible porque cada decisión se tomó en el lugar correcto:
    train aprende, validation decide, test estima al final.
"""

results_registry["student_facing_summary"] = student_facing_summary

print(student_facing_summary)

print_section("FIN DEL NOTEBOOK")
print("Validación, selección de modelos, KNN, SVM e imbalance")
print("Final audit passed:", NOTEBOOK_FINAL_AUDIT_PASSED)
print("Key tables are available in results_registry.")
print("Audits are available in audit_registry.")

---

**Fin del Notebook**

*Última actualización: 2026*  
*Autor: Adaptado y mejorado para el curso de Machine Learning*